In [23]:
!pip install transformers seqeval evaluate datasets

### Data Loading

In [24]:
from datasets import load_dataset

dataset_raw = load_dataset("lfcc/portuguese_ner")
dataset_raw

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})

In [25]:
dataset_raw["train"].features

{'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']))}

### Data Pre-Processing

In [26]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

In [27]:
inputs = tokenizer("As aulas de PLNEB são muito interessantes!")
inputs

{'input_ids': [101, 510, 6880, 125, 212, 22327, 22320, 19591, 453, 785, 20764, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [28]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(tokens)

['[CLS]', 'As', 'aulas', 'de', 'P', '##L', '##N', '##EB', 'são', 'muito', 'interessantes', '!', '[SEP]']


In [29]:
dataset_raw["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

In [30]:
tokens = ["as", "aulas", "plneb", "são", "interessantes", "!"]
inputs = tokenizer(tokens, is_split_into_words=True)

new_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
print(new_tokens)

['[CLS]', 'as', 'aulas', 'pl', '##ne', '##b', 'são', 'interessantes', '!', '[SEP]']


In [31]:
len(tokens), len(new_tokens)

(6, 10)

In [32]:
inputs.word_ids()

[None, 0, 1, 2, 2, 2, 3, 4, 5, None]

In [33]:
def align_labels_with_tokens(word_ids, labels):
    new_labels = []
    previous_word = None
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)
        elif previous_word != word_id:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        previous_word = word_id
    return new_labels


def tokenize_dataset(dataset):
    res = []
    for row in dataset:
        inputs = tokenizer(row["tokens"], is_split_into_words=True, truncation=True,max_length=512)
        new_labels = align_labels_with_tokens(inputs.word_ids(), row["ner_tags"])
        inputs["labels"] = new_labels[:len(inputs["input_ids"])]
        res.append(inputs)
    return res

train_dataset = tokenize_dataset(dataset_raw["train"])
test_dataset = tokenize_dataset(dataset_raw["test"])
len(train_dataset), len(test_dataset)


(3716, 930)

In [34]:
from datasets import Dataset
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


### Model Training

In [35]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("neuralmind/bert-base-portuguese-cased")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

TPC - treinar o modelo
(seguir o tutodial no https://huggingface.co/docs/transformers/tasks/token_classification), encontrar um textoe aplicar o modelo ao texto

In [36]:
label_list = dataset_raw["train"].features["ner_tags"].feature.names
label_list

['O',
 'B-Data',
 'I-Data',
 'B-Local',
 'I-Local',
 'B-Organizacao',
 'I-Organizacao',
 'B-Pessoa',
 'I-Pessoa',
 'B-Profissao',
 'I-Profissao']

In [37]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

### Evaluate

In [38]:
import evaluate

seqeval = evaluate.load("seqeval")

In [39]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### Train

In [40]:
id2label = {
    0: "O",
    1: "B-Data",
    2: "I-Data",
    3: "B-Local",
    4: "I-Local",
    5: "B-Organizacao",
    6: "I-Organizacao",
    7: "B-Pessoa",
    8: "I-Pessoa",
    9: "B-Profissao",
    10: "I-Profissao"
}

label2id = {
    "O": 0,
    "B-Data": 1,
    "I-Data": 2,
    "B-Local": 3,
    "I-Local": 4,
    "B-Organizacao": 5,
    "I-Organizacao": 6,
    "B-Pessoa": 7,
    "I-Pessoa": 8,
    "B-Profissao": 9,
    "I-Profissao": 10
}

In [41]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [42]:
training_args = TrainingArguments(
    output_dir="meu_modelo_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.076552,0.932861,0.958848,0.945676,0.981391
2,No log,0.068006,0.944616,0.966445,0.955406,0.984149


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=466, training_loss=0.14073420184876273, metrics={'train_runtime': 178.7141, 'train_samples_per_second': 41.586, 'train_steps_per_second': 2.608, 'total_flos': 436227558643800.0, 'train_loss': 0.14073420184876273, 'epoch': 2.0})

### Inference

In [43]:
texto_noticia="""O Conselho Médico dos Açores promoveu na passada sexta-feira, 24 de abril de 2026 uma cerimónia de homenagem aos médicos que se aposentaram ao longo de 2024 e 2025, assinalando o seu percurso de dedicação, excelência e serviço à saúde da população açoriana.
A sessão foi presidida pelo Presidente do Conselho Médico dos Açores, Carlos Ponte, que destacou o significado profundo deste reconhecimento, sublinhando que “vai muito além das palavras” e representa a gratidão institucional por uma vida dedicada ao cuidado do próximo e ao exercício rigoroso da Medicina.
O evento contou com a presença da Secretária Regional da Saúde e Segurança Social, Mónica Seidi, cuja participação foi destacada como um sinal de reconhecimento institucional pelo contributo destes profissionais à Região. Esteve também presente o Presidente do Conselho Médico da Região Sul, Paulo Simões, que se deslocou propositadamente de Lisboa, reforçando os laços de união entre os médicos a nível nacional.
Durante a cerimónia, foi salientado o papel determinante dos médicos no contexto atual, caracterizado por exigências crescentes nos sistemas de saúde. Apesar dos desafios, os profissionais agora homenageados demonstraram, ao longo das suas carreiras, elevada capacidade de adaptação, sentido de responsabilidade e compromisso com os doentes e comunidades. """

In [44]:
from transformers import pipeline

classifier = pipeline("ner",model=model, tokenizer=tokenizer, aggregation_strategy="first")
classifier(texto_noticia)


[{'entity_group': 'Local',
  'score': np.float32(0.65984774),
  'word': 'Açores',
  'start': 22,
  'end': 28},
 {'entity_group': 'Data',
  'score': np.float32(0.91423595),
  'word': '24 de abril de 2026',
  'start': 62,
  'end': 81},
 {'entity_group': 'Data',
  'score': np.float32(0.96696556),
  'word': '2024',
  'start': 152,
  'end': 156},
 {'entity_group': 'Data',
  'score': np.float32(0.9533638),
  'word': '2025',
  'start': 159,
  'end': 163},
 {'entity_group': 'Profissao',
  'score': np.float32(0.73150337),
  'word': 'Presidente',
  'start': 287,
  'end': 297},
 {'entity_group': 'Local',
  'score': np.float32(0.6901834),
  'word': 'Açores',
  'start': 321,
  'end': 327},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.9813087),
  'word': 'Carlos Ponte',
  'start': 329,
  'end': 341},
 {'entity_group': 'Profissao',
  'score': np.float32(0.6763154),
  'word': 'Secretária Regional da Saúde',
  'start': 601,
  'end': 629},
 {'entity_group': 'Profissao',
  'score': np.float32(0.49